<a href="https://colab.research.google.com/github/Madhura-123-p/FSDlab-1/blob/main/ADSLab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import deque


# linked list - stores the books a member has borrowed
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None


class LinkedList:
    def __init__(self):
        self.head = None

    def add(self, data):
        node = Node(data)
        node.next = self.head
        self.head = node

    def remove(self, data):
        prev = None
        cur = self.head
        while cur:
            if cur.data == data:
                if prev is None:
                    self.head = cur.next
                else:
                    prev.next = cur.next
                return True
            prev = cur
            cur = cur.next
        return False

    def contains(self, data):
        cur = self.head
        while cur:
            if cur.data == data:
                return True
            cur = cur.next
        return False

    def to_list(self):
        result = []
        cur = self.head
        while cur:
            result.append(cur.data)
            cur = cur.next
        return result


# queue - people waiting for a book
class Queue:
    def __init__(self):
        self.items = deque()

    def enqueue(self, item):
        self.items.append(item)

    def enqueue_front(self, item):
        self.items.appendleft(item)

    def dequeue(self):
        return self.items.popleft()

    def remove(self, item):
        self.items.remove(item)

    def is_empty(self):
        return len(self.items) == 0

    def contains(self, item):
        return item in self.items

    def to_list(self):
        return list(self.items)


# stack - keeps history of actions for undo
class Stack:
    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        return self.items.pop()

    def is_empty(self):
        return len(self.items) == 0


# binary search tree - keeps books sorted by title
class TreeNode:
    def __init__(self, key, book):
        self.key = key
        self.book = book
        self.left = None
        self.right = None


class BST:
    def __init__(self):
        self.root = None

    def get_key(self, book):
        return (book.title.lower(), book.isbn)

    def insert(self, book):
        self.root = self._insert(self.root, self.get_key(book), book)

    def _insert(self, node, key, book):
        if node is None:
            return TreeNode(key, book)
        if key < node.key:
            node.left = self._insert(node.left, key, book)
        elif key > node.key:
            node.right = self._insert(node.right, key, book)
        return node

    def delete(self, book):
        self.root = self._delete(self.root, self.get_key(book))

    def _delete(self, node, key):
        if node is None:
            return None
        if key < node.key:
            node.left = self._delete(node.left, key)
        elif key > node.key:
            node.right = self._delete(node.right, key)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            # two children: use smallest node on the right side
            small = node.right
            while small.left:
                small = small.left
            node.key = small.key
            node.book = small.book
            node.right = self._delete(node.right, small.key)
        return node

    def in_order(self):
        result = []
        self._in_order(self.root, result)
        return result

    def _in_order(self, node, result):
        if node:
            self._in_order(node.left, result)
            result.append(node.book)
            self._in_order(node.right, result)

    def search_title(self, word):
        word = word.lower()
        found = []
        for book in self.in_order():
            if word in book.title.lower():
                found.append(book)
        return found


class Book:
    def __init__(self, isbn, title, author, copies):
        self.isbn = isbn
        self.title = title
        self.author = author
        self.copies = copies
        self.available = copies

    def __str__(self):
        return f"[{self.isbn}] {self.title} by {self.author} ({self.available}/{self.copies} available)"


class Member:
    def __init__(self, member_id, name):
        self.member_id = member_id
        self.name = name
        self.borrowed = LinkedList()


class Library:
    def __init__(self):
        self.books = {}        # isbn -> Book
        self.members = {}      # member id -> Member
        self.catalog = BST()
        self.waitlists = {}    # isbn -> Queue
        self.history = Stack()

    def add_book(self, isbn, title, author, copies):
        if isbn in self.books:
            return "This ISBN already exists."
        book = Book(isbn, title, author, copies)
        self.books[isbn] = book
        self.catalog.insert(book)
        self.history.push(("add_book", isbn))
        return "Added: " + str(book)

    def remove_book(self, isbn):
        if isbn not in self.books:
            return "Book not found."
        book = self.books[isbn]
        if book.available != book.copies:
            return "Cannot remove, some copies are borrowed."
        del self.books[isbn]
        self.catalog.delete(book)
        old_queue = self.waitlists.pop(isbn, None)
        self.history.push(("remove_book", book, old_queue))
        return "Removed: " + book.title

    def add_member(self, member_id, name):
        if member_id in self.members:
            return "Member already exists."
        self.members[member_id] = Member(member_id, name)
        self.history.push(("add_member", member_id))
        return "Member added: " + name

    def issue_book(self, member_id, isbn):
        if member_id not in self.members or isbn not in self.books:
            return "Member or book not found."
        member = self.members[member_id]
        book = self.books[isbn]

        if member.borrowed.contains(isbn):
            return member.name + " already has this book."

        if book.available > 0:
            book.available -= 1
            member.borrowed.add(isbn)
            self.history.push(("issue", member_id, isbn))
            return f"Issued '{book.title}' to {member.name}."

        # no copy left so the member goes in the queue
        if isbn not in self.waitlists:
            self.waitlists[isbn] = Queue()
        queue = self.waitlists[isbn]
        if queue.contains(member_id):
            return "Already in the waiting list."
        queue.enqueue(member_id)
        self.history.push(("wait", member_id, isbn))
        return f"No copies left. {member.name} added to the waiting list."

    def return_book(self, member_id, isbn):
        if member_id not in self.members or isbn not in self.books:
            return "Member or book not found."
        member = self.members[member_id]
        book = self.books[isbn]

        if not member.borrowed.contains(isbn):
            return "This member has not borrowed this book."

        member.borrowed.remove(isbn)
        book.available += 1
        message = f"{member.name} returned '{book.title}'."

        # give it to the first person waiting, if any
        next_id = None
        queue = self.waitlists.get(isbn)
        if queue is not None and not queue.is_empty():
            next_id = queue.dequeue()
            self.members[next_id].borrowed.add(isbn)
            book.available -= 1
            message += " Issued to " + self.members[next_id].name + " from the waiting list."

        self.history.push(("return", member_id, isbn, next_id))
        return message

    def undo(self):
        if self.history.is_empty():
            return "Nothing to undo."

        action = self.history.pop()
        kind = action[0]

        if kind == "add_book":
            book = self.books.pop(action[1])
            self.catalog.delete(book)
            return "Undone: adding " + book.title

        if kind == "remove_book":
            book = action[1]
            old_queue = action[2]
            self.books[book.isbn] = book
            self.catalog.insert(book)
            if old_queue is not None:
                self.waitlists[book.isbn] = old_queue
            return "Undone: removing " + book.title

        if kind == "add_member":
            del self.members[action[1]]
            return "Undone: adding member"

        if kind == "issue":
            member_id = action[1]
            isbn = action[2]
            self.members[member_id].borrowed.remove(isbn)
            self.books[isbn].available += 1
            return "Undone: issuing book"

        if kind == "wait":
            member_id = action[1]
            isbn = action[2]
            self.waitlists[isbn].remove(member_id)
            return "Undone: waiting list entry"

        if kind == "return":
            member_id = action[1]
            isbn = action[2]
            next_id = action[3]
            book = self.books[isbn]
            # undo the automatic issue first, then the return
            if next_id is not None:
                self.members[next_id].borrowed.remove(isbn)
                book.available += 1
                self.waitlists[isbn].enqueue_front(next_id)
            self.members[member_id].borrowed.add(isbn)
            book.available -= 1
            return "Undone: returning book"


def ask_text(prompt):
    while True:
        value = input(prompt).strip()
        if value != "":
            return value
        print("This cannot be empty.")


def ask_number(prompt):
    while True:
        value = input(prompt).strip()
        if value.isdigit() and int(value) > 0:
            return int(value)
        print("Enter a number greater than 0.")


def main():
    lib = Library()

    while True:
        print("\n----- Library Menu -----")
        print("1. Add book")
        print("2. Remove book")
        print("3. Add member")
        print("4. Find book by ISBN")
        print("5. Search book by title")
        print("6. Show all books")
        print("7. Issue book")
        print("8. Return book")
        print("9. Show member's books")
        print("10. Show waiting list")
        print("11. Undo")
        print("0. Exit")
        choice = input("Enter choice: ").strip()

        if choice == "1":
            isbn = ask_text("ISBN: ")
            title = ask_text("Title: ")
            author = ask_text("Author: ")
            copies = ask_number("Copies: ")
            print(lib.add_book(isbn, title, author, copies))

        elif choice == "2":
            isbn = ask_text("ISBN: ")
            print(lib.remove_book(isbn))

        elif choice == "3":
            member_id = ask_text("Member ID: ")
            name = ask_text("Name: ")
            print(lib.add_member(member_id, name))

        elif choice == "4":
            isbn = ask_text("ISBN: ")
            book = lib.books.get(isbn)
            if book:
                print(book)
            else:
                print("Book not found.")

        elif choice == "5":
            word = ask_text("Title contains: ")
            results = lib.catalog.search_title(word)
            if len(results) == 0:
                print("No books found.")
            for book in results:
                print(book)

        elif choice == "6":
            books = lib.catalog.in_order()
            if len(books) == 0:
                print("No books in the library.")
            for book in books:
                print(book)

        elif choice == "7":
            member_id = ask_text("Member ID: ")
            isbn = ask_text("ISBN: ")
            print(lib.issue_book(member_id, isbn))

        elif choice == "8":
            member_id = ask_text("Member ID: ")
            isbn = ask_text("ISBN: ")
            print(lib.return_book(member_id, isbn))

        elif choice == "9":
            member_id = ask_text("Member ID: ")
            if member_id not in lib.members:
                print("Member not found.")
            else:
                member = lib.members[member_id]
                isbns = member.borrowed.to_list()
                if len(isbns) == 0:
                    print(member.name + " has no books.")
                for isbn in isbns:
                    print(lib.books[isbn])

        elif choice == "10":
            isbn = ask_text("ISBN: ")
            queue = lib.waitlists.get(isbn)
            if queue is None or queue.is_empty():
                print("Nobody is waiting.")
            else:
                count = 1
                for member_id in queue.to_list():
                    print(count, lib.members[member_id].name)
                    count += 1

        elif choice == "11":
            print(lib.undo())

        elif choice == "0":
            print("Bye!")
            break

        else:
            print("Invalid choice.")


main()



----- Library Menu -----
1. Add book
2. Remove book
3. Add member
4. Find book by ISBN
5. Search book by title
6. Show all books
7. Issue book
8. Return book
9. Show member's books
10. Show waiting list
11. Undo
0. Exit
Enter choice: 3
Member ID: Madhura Peshave
Name: Madhura
Member added: Madhura

----- Library Menu -----
1. Add book
2. Remove book
3. Add member
4. Find book by ISBN
5. Search book by title
6. Show all books
7. Issue book
8. Return book
9. Show member's books
10. Show waiting list
11. Undo
0. Exit
